<a href="https://colab.research.google.com/github/lamonega/colab-llm/blob/main/colab_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess
import time
import requests
import threading
import os
import shutil

MODEL_NAME = "qwen3:14b"

os.environ['OLLAMA_CONTEXT_LENGTH'] = '16384'
os.environ['OLLAMA_HOST'] = '0.0.0.0'
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'

def command_exists(cmd):
    """Verifica si un comando está disponible en el sistema."""
    return shutil.which(cmd) is not None

def ollama_responds():
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        return r.status_code == 200
    except:
        return False

# ------------------------------------------------------------
# 1. Validar / instalar curl
# ------------------------------------------------------------
if command_exists("curl"):
    print("[OK] curl ya instalado.")
else:
    print("Instalando curl...")
    subprocess.run(['apt-get', 'update'], capture_output=True, check=True)
    subprocess.run(['apt-get', 'install', '-y', 'curl'], capture_output=True, check=True)
    print("[OK] curl instalado.")

# ------------------------------------------------------------
# 2. Validar / instalar Ollama
# ------------------------------------------------------------
if command_exists("ollama"):
    print("[OK] Ollama ya instalado.")
else:
    print("Instalando Ollama...")
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
    print("[OK] Ollama instalado.")

# ------------------------------------------------------------
# 3. Validar / iniciar servicio Ollama
# ------------------------------------------------------------
if ollama_responds():
    print("[OK] El servicio Ollama ya está corriendo.")
else:
    print("Iniciando servicio Ollama...")
    def serve():
        subprocess.call(['ollama', 'serve'])
    hilo = threading.Thread(target=serve, daemon=True)
    hilo.start()

    for i in range(30):
        if ollama_responds():
            print(f"[OK] Servicio listo en {i+1}s.")
            break
        time.sleep(1)
    else:
        raise RuntimeError("[ERROR] Ollama no arrancó correctamente.")

# ------------------------------------------------------------
# 4. Validar / descargar modelo
# ------------------------------------------------------------
result = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
if MODEL_NAME in result.stdout:
    print(f"[OK] Modelo '{MODEL_NAME}' ya descargado.")
else:
    print(f"Descargando {MODEL_NAME}...")
    pull = subprocess.run(['ollama', 'pull', MODEL_NAME], capture_output=True, text=True)
    if pull.returncode != 0:
        raise RuntimeError(f"[ERROR] Falló descarga: {pull.stderr}")
    print(f"[OK] Modelo descargado: {MODEL_NAME}")

# ------------------------------------------------------------
# 5. Cargar modelo en VRAM
# ------------------------------------------------------------
print("Cargando modelo en VRAM...")
payload = {"model": MODEL_NAME, "prompt": "Hola", "stream": False}
r = requests.post("http://localhost:11434/api/generate", json=payload, timeout=300)
if r.status_code == 200:
    print("[OK] Todo listo. Modelo en VRAM.")
else:
    print(f"[ADVERTENCIA] Error al cargar modelo (codigo {r.status_code}), pero puede funcionar igual.")

In [ ]:
import subprocess
import re
import time
import requests
import os
import shutil

# ------------------------------------------------------------
# Obtener el nombre del modelo desde las variables de entorno
# ------------------------------------------------------------
model = os.environ.get('MODEL_NAME')
if not model:
    raise RuntimeError("No se encontró la variable MODEL_NAME. Ejecuta la Celda 1 primero.")

# ------------------------------------------------------------
# Validar / instalar wget (necesario para cloudflared)
# ------------------------------------------------------------
def command_exists(cmd):
    return shutil.which(cmd) is not None

if not command_exists('wget'):
    print("wget no instalado. Instalando...")
    subprocess.run(['apt-get', 'update'], capture_output=True, check=True)
    subprocess.run(['apt-get', 'install', '-y', 'wget'], capture_output=True, check=True)
    print("[OK] wget instalado.")
else:
    print("[OK] wget ya instalado.")

# ------------------------------------------------------------
# Paso 1: descargar cloudflared si no existe
# ------------------------------------------------------------
print("Verificando cloudflared...")
if os.path.exists('./cloudflared'):
    print("[OK] cloudflared ya existe.")
else:
    print("Descargando cloudflared...")
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', 'cloudflared'], check=True)
    subprocess.run(['chmod', '+x', 'cloudflared'], check=True)
    print("[OK] cloudflared listo.")

# ------------------------------------------------------------
# Verificar que el servicio Ollama local esté corriendo antes del túnel
# ------------------------------------------------------------
def ollama_local_responds():
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        return r.status_code == 200
    except:
        return False

if not ollama_local_responds():
    raise RuntimeError("El servicio Ollama local no responde en http://localhost:11434. Asegúrate de que esté corriendo antes de iniciar el túnel.")

print("[OK] Servicio Ollama local está activo.")

# ------------------------------------------------------------
# Verificar si ya hay un proceso cloudflared corriendo (para evitar duplicados)
# ------------------------------------------------------------
def hay_tunel_activo():
    try:
        # Buscar procesos cloudflared que no sean este script
        result = subprocess.run(['pgrep', '-f', 'cloudflared tunnel'], capture_output=True, text=True)
        return result.returncode == 0
    except:
        return False

if hay_tunel_activo():
    print("[ADVERTENCIA] Parece que ya hay un túnel cloudflared corriendo. Se intentará detenerlo...")
    subprocess.run(['pkill', '-f', 'cloudflared tunnel'], capture_output=True)
    time.sleep(2)

# ------------------------------------------------------------
# Paso 2: iniciar el túnel y obtener la URL pública
# ------------------------------------------------------------
print("Iniciando túnel de Cloudflare...")
proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

public_url = None
inicio = time.time()

# Leer la salida del túnel línea por línea hasta encontrar la URL
for linea in proc.stdout:
    if "https://" in linea and "trycloudflare.com" in linea:
        coincidencia = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', linea)
        if coincidencia:
            public_url = coincidencia.group(1)
            break
    if time.time() - inicio > 60:
        proc.terminate()
        raise RuntimeError("No se obtuvo la URL pública en 60 segundos.")

if not public_url:
    proc.terminate()
    raise RuntimeError("No se pudo extraer la URL pública.")

print("[OK] Túnel creado. URL obtenida.")

# ------------------------------------------------------------
# Paso 3: esperar que la URL pública sea accesible
# ------------------------------------------------------------
print("Verificando que la URL pública responda...")
for intento in range(1, 13):
    try:
        respuesta = requests.get(f"{public_url}/api/tags", timeout=5)
        if respuesta.status_code == 200:
            print(f"[OK] URL pública accesible (intento {intento}).")
            break
    except:
        pass
    time.sleep(5)
else:
    proc.terminate()
    raise RuntimeError("La URL pública no responde después de varios intentos.")

# ------------------------------------------------------------
# Paso 4: verificar que el modelo exista localmente antes de probar
# ------------------------------------------------------------
result = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
if model not in result.stdout:
    proc.terminate()
    raise RuntimeError(f"El modelo '{model}' no está descargado localmente. Descárgalo con 'ollama pull {model}' antes de continuar.")

print(f"[OK] Modelo '{model}' está disponible localmente.")

# ------------------------------------------------------------
# Paso 5: hacer una prueba sencilla al modelo
# ------------------------------------------------------------
print("Enviando pregunta de prueba al modelo...")
payload = {
    "model": model,
    "prompt": "¿Cuál es el resultado de sumar 2 + 2?",
    "stream": False
}

try:
    r = requests.post(f"{public_url}/api/generate", json=payload, timeout=120)
    r.raise_for_status()
    datos = r.json()
    if datos.get('response'):
        print("[OK] Modelo responde correctamente.")
    else:
        raise RuntimeError("La respuesta del modelo está vacía.")
except Exception as e:
    proc.terminate()
    raise RuntimeError(f"Error al consultar el modelo: {e}")

# ------------------------------------------------------------
# Paso 6: mostrar la URL pública
# ------------------------------------------------------------
print("------------------------------------------------------------")
print("TODO LISTO. El servicio está activo y accesible desde internet.")
print(f"URL pública: {public_url}")
print("------------------------------------------------------------")